# Normalizacja metadanych produktów (LLM)

**Etap 1** — LLM analizuje wszystkie unikalne wartości z każdej kolumny i tworzy canonical mapping  
**Etap 2** — LLM mapuje wartości każdego wiersza na canonical (używając listy z etapu 1)  
**Output** → `zurada_products_with_metadata_cleaned.csv`

Checkpoint po etapie 2 — wznawialne.

In [1]:
%pip install openai pandas tqdm --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, time
import pandas as pd
from collections import Counter
from tqdm.notebook import tqdm
from openai import OpenAI

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "Brak klucza API")  # ustaw zmienną środowiskową lub wklej tu
MODEL          = "gpt-5.5"
INPUT_CSV      = "extended_zurada_products_with_metadata.csv"
CANONICAL_JSON = "extended_zurada_canonical_mappings.json"   # wynik etapu 1
CHECKPOINT_CSV = "extended_zurada_normalize_checkpoint.csv"  # wynik etapu 2
OUTPUT_CSV     = "extended_zurada_products_with_metadata_cleaned.csv"
SLEEP_BETWEEN  = 0.3
MAX_RETRIES    = 3

LIST_COLS   = ["dozwolone_powierzchnie", "odradzane_powierzchnie",
               "metoda_mycia", "zgodnosc_i_certyfikaty", "ostrzezenia_bhp"]
STRING_COLS = ["odczyn_ph"]  # te mają prostą wartość, nie listę
ALL_COLS    = LIST_COLS + STRING_COLS

client = OpenAI(api_key=OPENAI_API_KEY)

PHASE1_SYSTEM = """Jesteś ekspertem normalizacji danych produktowych.
Dostajesz listę unikalnych wartości z jednej kolumny CSV produktów chemicznych.
Twoim zadaniem jest:
1. Zgrupować podobne/synonimiczne wartości
2. Dla każdej grupy wybrać jedną kanoniczną formę (canonical value) - krótką, polską, spójną
3. Stworzyć mapping: każda oryginalna wartość → jej canonical odpowiednik
Canonical values powinny być: po polsku, małymi literami, krótkie (1-4 słowa), spójne.
Odpowiadasz wyłącznie poprawnym JSON."""

print(f"Model: {MODEL}")

Model: gpt-5.5


In [3]:
df = pd.read_csv(INPUT_CSV)
print(f"Wczytano {len(df)} produktów")


def parse_val(val, is_list=True):
    """Parsuje wartość z CSV (JSON string lub plain string)."""
    if pd.isna(val) or str(val).strip() == "":
        return [] if is_list else ""
    try:
        parsed = json.loads(val)
        return parsed if is_list else str(parsed)
    except Exception:
        return [str(val)] if is_list else str(val)


def collect_unique(col, is_list=True):
    """Zbiera wszystkie unikalne wartości z kolumny."""
    counter = Counter()
    for raw in df[col].dropna():
        items = parse_val(raw, is_list)
        if is_list:
            counter.update(items)
        else:
            counter[items] += 1
    return counter


print("Funkcje pomocnicze gotowe.")

Wczytano 262 produktów
Funkcje pomocnicze gotowe.


## Etap 1 — Generowanie canonical mappings przez LLM

In [4]:
def generate_canonical_mapping(col_name, unique_counter):
    items = list(unique_counter.keys())
    freq_lines = "\n".join(f"{unique_counter[v]:3d}x  {v}" for v in items)

    col_context = {
        "dozwolone_powierzchnie":  "powierzchnie/materiały, na których produkt MOŻE być stosowany",
        "odradzane_powierzchnie":  "powierzchnie/materiały, na których produkt NIE POWINIEN być stosowany",
        "metoda_mycia":            "metody aplikacji/mycia (ręczne, maszynowe, ciśnieniowe itp.)",
        "zgodnosc_i_certyfikaty":  "certyfikaty, normy i zgodności produktu",
        "ostrzezenia_bhp":         "ostrzeżenia i instrukcje bezpieczeństwa",
        "odczyn_ph":               "odczyn pH produktu",
    }.get(col_name, col_name)

    user_msg = (
        f"Kolumna: {col_name} ({col_context})\n"
        f"Lista {len(items)} unikalnych wartości (z częstotliwością):\n"
        f"{freq_lines}\n\n"
        'Zwróć JSON: {"canonical_values": [...], "mapping": {"oryg": "canonical", ...}}'
    )

    for attempt in range(1, MAX_RETRIES + 1):
        print(f"    [attempt {attempt}] wysyłam {len(items)} wartości do modelu...")
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": PHASE1_SYSTEM},
                    {"role": "user",   "content": user_msg},
                ],
                response_format={"type": "json_object"},
                max_completion_tokens=60000,
            )
            content = response.choices[0].message.content
            if not content:
                refusal = getattr(response.choices[0].message, "refusal", None)
                raise ValueError(f"Pusta odpowiedź. Refusal: {refusal}")
            parsed = json.loads(content)
            canonical = parsed.get("canonical_values", [])
            mapping   = parsed.get("mapping", {})
            print(f"    OK — {len(canonical)} canonical values, {len(mapping)}/{len(items)} mappings")
            return {"canonical_values": canonical, "mapping": mapping}
        except Exception as e:
            wait = 2 ** attempt
            print(f"    [RETRY {attempt}/{MAX_RETRIES}] {col_name} — {e} — czekam {wait}s")
            time.sleep(wait)

    raise RuntimeError(f"Nie udało się wygenerować mappingu dla {col_name}")


print("Funkcja generate_canonical_mapping gotowa.")


Funkcja generate_canonical_mapping gotowa.


In [5]:
# Wczytaj checkpoint jeśli istnieje
if os.path.exists(CANONICAL_JSON):
    with open(CANONICAL_JSON, "r", encoding="utf-8") as f:
        all_mappings = json.load(f)
    print(f"Wczytano istniejące mappingi: {list(all_mappings.keys())}")
else:
    all_mappings = {}
    print("Brak mappingów — generuję od zera.")

for col in ALL_COLS:
    if col in all_mappings:
        print(f"  Pomijam {col} (już wygenerowane)")
        continue

    is_list = col in LIST_COLS
    counter = collect_unique(col, is_list=is_list)
    print(f"  Generuję mapping dla '{col}' ({len(counter)} unikalnych wartości)...")

    result = generate_canonical_mapping(col, counter)
    all_mappings[col] = result

    # Zapis po każdej kolumnie
    with open(CANONICAL_JSON, "w", encoding="utf-8") as f:
        json.dump(all_mappings, f, ensure_ascii=False, indent=2)

    print(f"    → {len(result['canonical_values'])} canonical values, {len(result['mapping'])} mappings")
    time.sleep(SLEEP_BETWEEN)

print("\nEtap 1 zakończony. Mappingi zapisane w:", CANONICAL_JSON)

Brak mappingów — generuję od zera.
  Generuję mapping dla 'dozwolone_powierzchnie' (453 unikalnych wartości)...
    [attempt 1] wysyłam 453 wartości do modelu...
    OK — 307 canonical values, 453/453 mappings
    → 307 canonical values, 453 mappings
  Generuję mapping dla 'odradzane_powierzchnie' (66 unikalnych wartości)...
    [attempt 1] wysyłam 66 wartości do modelu...
    OK — 44 canonical values, 66/66 mappings
    → 44 canonical values, 66 mappings
  Generuję mapping dla 'metoda_mycia' (40 unikalnych wartości)...
    [attempt 1] wysyłam 40 wartości do modelu...
    OK — 22 canonical values, 40/40 mappings
    → 22 canonical values, 40 mappings
  Generuję mapping dla 'zgodnosc_i_certyfikaty' (27 unikalnych wartości)...
    [attempt 1] wysyłam 27 wartości do modelu...
    OK — 24 canonical values, 27/27 mappings
    → 24 canonical values, 27 mappings
  Generuję mapping dla 'ostrzezenia_bhp' (354 unikalnych wartości)...
    [attempt 1] wysyłam 354 wartości do modelu...
    OK — 60 

In [6]:
# Podgląd wygenerowanych canonical values
for col, data in all_mappings.items():
    print(f"\n=== {col} ({len(data['canonical_values'])} canonical) ===")
    for v in sorted(data["canonical_values"]):
        print(f"  • {v}")


=== dozwolone_powierzchnie (307 canonical) ===
  • akcesoria kuchenne
  • akcesoria skórzane
  • aluminium
  • aluminium polerowane
  • armatura
  • autobusy
  • balustrady
  • bemary
  • beton
  • bez kontaktu z żywnością
  • blachy
  • blaty
  • bojlery
  • brodziki
  • burty naczep
  • centra logistyczne
  • ceramika
  • ceramika sanitarna
  • ciepłe powierzchnie
  • ciężarówki
  • cysterny
  • czajniki
  • czyste elementy
  • części maszyn
  • części metalowe
  • części samochodowe
  • deski rozdzielcze
  • drewno
  • drewno lakierowane
  • duże powierzchnie
  • dystrybutory paliwa
  • dywaniki
  • dywany
  • ekrany lcd
  • ekrany led
  • ekrany oled
  • ekspresy do kawy
  • elementy chromowane
  • elementy gwintowane
  • elementy mechaniczne
  • elementy montażowe
  • elementy plastikowe
  • elewacje
  • epoksyd
  • fasady
  • felgi
  • felgi aluminiowe
  • felgi lakierowane
  • felgi stalowe
  • fermentatory
  • filtry dpf
  • filtry fap
  • filtry spalin
  • folie zabezpieczają

## Etap 2 — Mapowanie każdego wiersza przez LLM

In [7]:
def build_phase2_schema(col, is_list):
    canonical_vals = all_mappings[col]["canonical_values"]
    enum_values = canonical_vals + [""]  # pusty string dla brak wartości

    if is_list:
        return {
            "type": "object",
            "properties": {
                "values": {
                    "type": "array",
                    "items": {"type": "string", "enum": enum_values},
                    "description": f"Zmapowane wartości kolumny {col} na canonical. Puste [] jeśli brak."
                }
            },
            "required": ["values"],
            "additionalProperties": False
        }
    else:
        return {
            "type": "object",
            "properties": {
                "value": {
                    "type": "string",
                    "enum": enum_values,
                    "description": f"Zmapowana wartość kolumny {col} na canonical. Pusty string jeśli brak."
                }
            },
            "required": ["value"],
            "additionalProperties": False
        }


PHASE2_SYSTEM = """Jesteś asystentem normalizacji danych.
Dostajesz listę surowych wartości z jednej kolumny produktu i listę dozwolonych canonical values.
Zmapuj każdą surową wartość na najbliższy canonical odpowiednik.
Jeśli wartość nie pasuje do żadnej canonical — usuń ją (nie zwracaj jej).
Nie dodawaj wartości spoza canonical_values.
Odpowiadasz wyłącznie poprawnym JSON."""


def map_row_column(raw_val, col, is_list):
    """Mapuje wartości jednej kolumny jednego wiersza na canonical."""
    parsed = parse_val(raw_val, is_list)

    # Szybka ścieżka: jeśli puste — zwróć puste
    if not parsed or parsed == []:
        return [] if is_list else ""

    canonical_vals = all_mappings[col]["canonical_values"]
    mapping = all_mappings[col]["mapping"]

    # Szybka ścieżka: spróbuj programistycznego mappingu najpierw
    if is_list:
        result = []
        needs_llm = []
        for v in parsed:
            if v in mapping and mapping[v] in canonical_vals:
                result.append(mapping[v])
            else:
                needs_llm.append(v)
    else:
        if parsed in mapping and mapping[parsed] in canonical_vals:
            return mapping[parsed]
        needs_llm = [parsed]
        result = []

    # Jeśli wszystko zmapowane programistycznie — nie wywołuj LLM
    if not needs_llm:
        return list(dict.fromkeys(result)) if is_list else (result[0] if result else "")

    # Wywołaj LLM dla wartości bez mappingu
    schema = build_phase2_schema(col, is_list)
    canonical_str = ", ".join(f'"{v}"' for v in canonical_vals)
    raw_str = json.dumps(needs_llm if is_list else needs_llm[0], ensure_ascii=False)

    user_msg = f"""Kolumna: {col}
Dozwolone canonical values: [{canonical_str}]

Surowe wartości do zmapowania: {raw_str}

Zmapuj na canonical. Usuń to, co nie pasuje."""

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": PHASE2_SYSTEM},
                    {"role": "user",   "content": user_msg},
                ],
                response_format={
                    "type": "json_schema",
                    "json_schema": {
                        "name": "row_mapping",
                        "strict": True,
                        "schema": schema,
                    }
                },
                max_completion_tokens=500,
            )
            llm_result = json.loads(response.choices[0].message.content)
            if is_list:
                llm_vals = [v for v in llm_result.get("values", []) if v and v in canonical_vals]
                combined = result + llm_vals
                return list(dict.fromkeys(combined))  # dedup zachowując kolejność
            else:
                v = llm_result.get("value", "")
                return v if v in canonical_vals else ""
        except Exception as e:
            wait = 2 ** attempt
            print(f"    [RETRY {attempt}] col={col} — {e} — czekam {wait}s")
            time.sleep(wait)

    # Fallback: zwróć tylko programistycznie zmapowane
    return list(dict.fromkeys(result)) if is_list else (result[0] if result else "")


print("Funkcja map_row_column gotowa (hybrydowa: mapping dict + LLM dla reszty).")

Funkcja map_row_column gotowa (hybrydowa: mapping dict + LLM dla reszty).


In [8]:
# ── Główna pętla etapu 2 ─────────────────────────────────────────────────────
if os.path.exists(CHECKPOINT_CSV):
    done_df  = pd.read_csv(CHECKPOINT_CSV)
    done_ids = set(done_df["product_id"].astype(int).tolist())
    print(f"Checkpoint: {len(done_ids)} produktów już przetworzonych.")
else:
    done_df  = pd.DataFrame()
    done_ids = set()
    print("Brak checkpointu — zaczynam od początku.")

to_process = df[~df["product_id"].astype(int).isin(done_ids)]
print(f"Do przetworzenia: {len(to_process)} produktów.")

results = []

for _, row in tqdm(to_process.iterrows(), total=len(to_process), desc="Normalizacja wierszy"):
    record = {"product_id": int(row["product_id"])}

    for col in LIST_COLS:
        mapped = map_row_column(row.get(col), col, is_list=True)
        record[col] = json.dumps(mapped, ensure_ascii=False)

    for col in STRING_COLS:
        mapped = map_row_column(row.get(col), col, is_list=False)
        record[col] = mapped

    results.append(record)
    time.sleep(SLEEP_BETWEEN)

    if len(results) % 5 == 0:
        batch = pd.DataFrame(results)
        combined = pd.concat([done_df, batch], ignore_index=True) if not done_df.empty else batch
        combined.to_csv(CHECKPOINT_CSV, index=False)

if results:
    batch = pd.DataFrame(results)
    done_df = pd.concat([done_df, batch], ignore_index=True) if not done_df.empty else batch
    done_df.to_csv(CHECKPOINT_CSV, index=False)

print(f"\nEtap 2 zakończony. {len(done_df)} produktów w checkpoincie.")

Brak checkpointu — zaczynam od początku.
Do przetworzenia: 262 produktów.


Normalizacja wierszy:   0%|          | 0/262 [00:00<?, ?it/s]

    [RETRY 1] col=ostrzezenia_bhp — Expecting value: line 1 column 1 (char 0) — czekam 2s
    [RETRY 1] col=ostrzezenia_bhp — Error code: 400 - {'error': {'message': 'Could not finish the message because max_tokens or model output limit was reached. Please try again with higher max_tokens.', 'type': 'invalid_request_error', 'param': None, 'code': None}} — czekam 2s
    [RETRY 1] col=ostrzezenia_bhp — Expecting value: line 1 column 1 (char 0) — czekam 2s
    [RETRY 1] col=ostrzezenia_bhp — Expecting value: line 1 column 1 (char 0) — czekam 2s
    [RETRY 2] col=ostrzezenia_bhp — Expecting value: line 1 column 1 (char 0) — czekam 4s
    [RETRY 1] col=ostrzezenia_bhp — Expecting value: line 1 column 1 (char 0) — czekam 2s
    [RETRY 1] col=ostrzezenia_bhp — Expecting value: line 1 column 1 (char 0) — czekam 2s
    [RETRY 1] col=ostrzezenia_bhp — Expecting value: line 1 column 1 (char 0) — czekam 2s
    [RETRY 1] col=ostrzezenia_bhp — Expecting value: line 1 column 1 (char 0) — czekam 2s

E

In [9]:
# ── Scalenie z oryginalnym CSV i zapis ───────────────────────────────────────
clean_meta = pd.read_csv(CHECKPOINT_CSV)
clean_meta["product_id"] = clean_meta["product_id"].astype(int)

# Zacznij od oryginalnych kolumn (bez starych metadanych)
orig_cols = [c for c in df.columns if c not in ALL_COLS]
base = df[orig_cols].copy()
base["product_id"] = base["product_id"].astype(int)

merged = base.merge(clean_meta, on="product_id", how="left")
merged.to_csv(OUTPUT_CSV, index=False)

print(f"Zapisano {len(merged)} wierszy do '{OUTPUT_CSV}'")
merged[["product_id", "product_name"] + ALL_COLS].head(3)

Zapisano 262 wierszy do 'extended_zurada_products_with_metadata_cleaned.csv'


,product_id,product_name,dozwolone_powierzchnie,odradzane_powierzchnie,metoda_mycia,zgodnosc_i_certyfikaty,ostrzezenia_bhp,odczyn_ph
0,1,Zurada Lśniąca Pianka,"[""wszystkie powierzchnie"", ""tworzywa sztuczne""...",[],"[""ręczne"", ""natryskowe""]",[],"[""spłukać wodą"", ""spłukać i osuszyć""]",NaN
1,2,Zurada Szklany Blask,"[""szkło"", ""gładkie powierzchnie"", ""porcelana"",...","[""powierzchnie wrażliwe na alkohol"", ""rozgrzan...","[""ręczne"", ""natryskowe"", ""pianowe""]","[""ifra"", ""rspo""]","[""unikać alkoholu"", ""unikać gorąca"", ""wykonać ...",NaN
2,3,Zurada Tłuszcz Stop,"[""blaty"", ""podłogi"", ""ściany"", ""naczynia"", ""ga...","[""aluminium"", ""miedź"", ""powierzchnie lakierowa...","[""pianowe"", ""ręczne""]",[],"[""dobrać stężenie"", ""wykonać próbę wstępną""]",NaN


In [50]:
# ── Walidacja: nowe rozkłady wartości ────────────────────────────────────────
from collections import Counter

for col in LIST_COLS:
    counter = Counter()
    for v in merged[col].dropna():
        try:
            counter.update(json.loads(v))
        except Exception:
            pass
    print(f"\n=== {col} ({len(counter)} canonical) ===")
    for val, cnt in counter.most_common(15):
        print(f"  {cnt:3d}x  {val}")

print(f"\n=== odczyn_ph ===")
print(merged["odczyn_ph"].value_counts().to_string())


=== dozwolone_powierzchnie (201 canonical) ===
   28x  tworzywa sztuczne
   21x  stal nierdzewna
   19x  szkło
   19x  powierzchnie ponadpodłogowe
   17x  podłogi
   14x  porcelana
   14x  zmywalne powierzchnie
   12x  naczynia
    9x  garnki
    9x  ceramika
    9x  powierzchnie sanitarne
    9x  wanny
    9x  gres
    9x  lakier
    8x  meble

=== odradzane_powierzchnie (48 canonical) ===
   25x  aluminium
   14x  powierzchnie wrażliwe na wodę
   12x  miedź
   10x  powierzchnie malowane
   10x  powierzchnie lakierowane
    8x  mosiądz
    7x  powierzchnie wrażliwe na zasady
    5x  powierzchnie wrażliwe na alkohol
    5x  rozgrzane powierzchnie
    5x  powierzchnie zabezpieczone polimerem
    5x  powierzchnie wrażliwe na kwasy
    4x  drewno
    4x  panele
    3x  piekarniki z automatycznym myciem
    3x  zmywarki domowe

=== metoda_mycia (23 canonical) ===
   80x  ręczne
   40x  pianowe
   26x  maszynowe
   22x  ciśnieniowe
   20x  natryskowe
    6x  systemy dozujące
    5x  CIP
  

In [51]:
# ── Podgląd konkretnego produktu ─────────────────────────────────────────────
PRODUCT_ID = 45

row = merged[merged["product_id"] == PRODUCT_ID].iloc[0]
print(f"=== {row['product_name']} (ID {PRODUCT_ID}) ===")
for col in ALL_COLS:
    val = row[col]
    try:
        val = json.loads(val)
    except Exception:
        pass
    print(f"\n{col}:")
    if isinstance(val, list):
        for item in val:
            print(f"  • {item}")
    else:
        print(f"  {val}")

=== Zurada Odtłuszczacz Moc (ID 45) ===

dozwolone_powierzchnie:
  • blaty
  • podłogi
  • ściany
  • naczynia
  • narzędzia
  • części samochodowe
  • zatłuszczone powierzchnie
  • sufity
  • przedmioty
  • wrażliwe na alkalia
  • aluminium
  • kontakt z żywnością

odradzane_powierzchnie:

metoda_mycia:
  • ręczne
  • ciśnieniowe
  • maszynowe
  • CIP

zgodnosc_i_certyfikaty:
  • do przemysłu spożywczego

ostrzezenia_bhp:
  • spłukać wodą pitną
  • wykonać próbę wstępną

odczyn_ph:
  nan
